# scRNA-seq pathway enrichment analysis — malignant cells (t-stat ranked GSEA)

**Goal:** Identify pathways enriched in MHC II high vs low malignant cells in the
Salcher scRNA-seq atlas using GSEA on a pre-ranked t-stat gene list. Results are
compared to the CosMx t-stat GSEA (`cancer_cell_pathway_analysis_tstat.ipynb`)
to identify concordantly enriched pathways across modalities.

**Input:** `luad_mhc2_classified.processed` — Salcher atlas malignant cells with
MHC II high/low classification; `data/cosmx_pathway_annotations.csv`.

**Output:** `outputs/tables/figure3/scrna_gsea_hallmark_t_test.csv`;
`outputs/tables/figure3/scrna_gsea_cosmx_pathways_t_test.csv`

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

import anndata as ad
import scanpy as sc
import decoupler as dc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
# the scRNA malignant cells file is in scratch, not outputs/processed
scrna_path         = Path(cfg['datasets']['luad_mhc2_classified']['processed'])
pathway_annot_path = repo_root / cfg['datasets']['cosmx']['pathway_annotations']

# outputs
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'
table_out = table_dir / 'figure3'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

Malignant cells from the Salcher atlas with MHC II high/low classification are
loaded. The var index is set to gene symbols (`feature_name`) to enable gene
set matching against Hallmark and CosMx pathway collections.

In [ ]:
import warnings

adata = ad.read_h5ad(scrna_path)
print(f"scRNA malignant cells: {adata.n_obs:,} × {adata.n_vars:,} genes")
print(adata.obs['MHC2_clustering'].value_counts())

# set var index to gene symbols for gene set matching
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    adata.var_names = adata.var['feature_name'].values
print(f"\nVar index (first 5): {adata.var_names[:5].tolist()}")

## Load gene sets

Same two gene set collections as the CosMx analysis — Hallmark and NanoString
CosMx pathway annotations — enabling direct comparison of enrichment results
across modalities.

In [ ]:
# load NanoString CosMx pathway annotations
pathways = pd.read_csv(pathway_annot_path, index_col=0)

long_df = (
    pathways.reset_index()
    .rename(columns={'Gene': 'genesymbol'})
    .melt(id_vars='genesymbol', var_name='geneset', value_name='Belongs')
    .query('Belongs == 1')
    .drop(columns='Belongs')
    .drop_duplicates(subset=['genesymbol', 'geneset'])
    .reset_index(drop=True)
)
long_df['collection'] = 'CosMx'
long_df['geneset'] = 'CosMx: ' + long_df['geneset'].astype(str)
print(f"CosMx pathway sets: {long_df['geneset'].nunique()}")

# load Hallmark gene sets
hallmark = dc.op.hallmark(organism='human', license='academic', verbose=False)
hallmark = hallmark[~hallmark.duplicated(['source', 'target'])]
print(f"Hallmark gene sets: {hallmark['source'].nunique()}")

## Subset to MHC II high vs low

Cells are subset to MHC II high and low classifications. The clonal group
is excluded from this comparison as in the CosMx analysis.

In [ ]:
high_low = adata[
    adata.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low'])
].copy()
print(f"High + low: {high_low.n_obs:,} cells")
print(high_low.obs['MHC2_clustering'].value_counts())

## Rank genes by differential expression

Genes are ranked by t-test scores comparing MHC II high vs low malignant cells
in the Salcher scRNA-seq atlas. The top 4000 highly variable genes are used to
focus the analysis on informative features while keeping the gene set broad enough
for pathway coverage — the full atlas has ~17,000 genes so HVG filtering is
necessary here unlike the CosMx analysis. Scores are rank-transformed before
GSEA for numerical stability.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sc.tl.rank_genes_groups(
        high_low,
        groupby='MHC2_clustering',
        method='t-test',
        key_added='t-test',
        use_raw=False,
    )

# use full gene ranking — scRNA has many more genes so we apply HVG filter
sc.pp.highly_variable_genes(
    high_low, n_top_genes=4000, flavor='seurat', subset=False,
)
hvg_col = 'highly_variable' if 'highly_variable' in high_low.var.columns else 'is_highly_variable'
hvg = high_low.var.index[high_low.var[hvg_col].astype(bool)].tolist()

t_stats = (
    sc.get.rank_genes_groups_df(high_low, 'MHC class II High', key='t-test')
    .set_index('names')
    .loc[lambda df: df.index.isin(hvg)]
    .sort_values('scores', key=np.abs, ascending=False)[['scores']]
)

t_stats['scores'] = (
    t_stats['scores'].rank(method='average')
    - t_stats['scores'].rank(method='average').mean()
)
t_stats['scores'] /= t_stats['scores'].std()

print(f"Genes ranked: {len(t_stats)}")
print(t_stats.head())

## GSEA helper functions

Same functions as in `cancer_cell_pathway_analysis` (CosMx) — `run_gsea_collection`
runs `dc.mt.gsea` and stores results in `adata.obsm`; `extract_gsea_results`
aggregates per-cell scores to group level using Mann-Whitney U test.

In [ ]:
def run_gsea_tstat(t_stats_df, net_df, collection_name, times=200, min_genes_in_set=5):
    """Run GSEA on a pre-ranked t-stat gene list."""
    if 'geneset' in net_df.columns:
        net = net_df.rename(columns={'geneset': 'source', 'genesymbol': 'target'})
    else:
        net = net_df.copy()

    net = net[~net.duplicated(['source', 'target'])]
    valid_sets = (
        net.groupby('source')['target']
        .nunique()
        .loc[lambda x: x >= min_genes_in_set]
        .index
    )
    net = net[net['source'].isin(valid_sets)]
    print(f"Running t-stat GSEA for {collection_name}: {net['source'].nunique()} gene sets")

    scores, pvals = dc.mt.gsea(
        t_stats_df.T,
        net=net,
        tmin=min_genes_in_set,
        times=times,
        seed=1,
        verbose=False,
    )

    results = pd.DataFrame({
        'NES': scores.iloc[0],
        'pval': pvals.iloc[0],
    })
    results['padj'] = multipletests(results['pval'], method='fdr_bh')[1]
    results['-log10_padj'] = -np.log10(results['padj'].replace(0, np.nextafter(0, 1)))
    results = results.sort_values('padj')

    return results

## Run GSEA

GSEA is run for Hallmark and CosMx pathway collections. Results are saved
as checkpoints — subsequent runs load from CSV rather than recomputing.
Note: Hallmark must be extracted before running CosMx since both overwrite
`adata.obsm['score_gsea']`.

In [ ]:
%%time

hallmark_results_path = table_out / 'scrna_gsea_hallmark_t_test.csv'
cosmx_results_path    = table_out / 'scrna_gsea_cosmx_pathways_t_test.csv'

if hallmark_results_path.exists():
    print("Loading pre-computed Hallmark t-stat GSEA results...")
    gsea_hallmark = pd.read_csv(hallmark_results_path, index_col=0)
else:
    print("Running Hallmark t-stat GSEA...")
    gsea_hallmark = run_gsea_tstat(
        t_stats, hallmark, 'hallmark', times=1000, min_genes_in_set=5
    )
    gsea_hallmark.to_csv(hallmark_results_path)
    print(f"Saved → {hallmark_results_path}")

if cosmx_results_path.exists():
    print("Loading pre-computed CosMx t-stat GSEA results...")
    gsea_cosmx = pd.read_csv(cosmx_results_path, index_col=0)
else:
    print("Running CosMx t-stat GSEA...")
    gsea_cosmx = run_gsea_tstat(
        t_stats, long_df, 'CosMx', times=1000, min_genes_in_set=5
    )
    gsea_cosmx.to_csv(cosmx_results_path)
    print(f"Saved → {cosmx_results_path}")

print("\nTop Hallmark pathways (t-stat):")
print(gsea_hallmark.sort_values('padj').head(10))

## Visualization

Pathway enrichment results are visualized as a barplot ranked by absolute NES
and a volcano plot of NES vs -log10(FDR). Both plots use the paper color palette
— orange for MHC II high/positive, purple for MHC II low/negative.

In [ ]:
def plot_gsea_barplot(gsea_df, top_n=25, title=None, figsize=(10, 10)):
    plot_df = (
        gsea_df.copy()
        .assign(Direction=np.where(gsea_df['NES'] >= 0, 'MHC-II High', 'MHC-II Low'))
        .assign(absNES=lambda d: d['NES'].abs())
        .sort_values('absNES', ascending=False)
        .head(top_n)
        .sort_values('NES')
    )
    # clean up pathway names
    plot_df.index = plot_df.index.str.replace('_', ' ').str.title()

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=plot_df, y=plot_df.index, x='NES',
        hue='Direction', dodge=False,
        palette={'MHC-II High': cmap_high_low[0], 'MHC-II Low': cmap_high_low[1]},
        edgecolor='none', ax=ax,
    )
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Normalized Enrichment Score (NES)')
    ax.set_ylabel('')
    ax.set_title(title or f'Top {top_n} Pathways by |NES|')
    ax.legend(title='Enriched In', bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig


def plot_gsea_volcano(gsea_df, sig_thresh=0.05, nes_thresh=None,
                      top_n_labels=15, title=None, figsize=(10, 8), ymax=30):
    
    score_col = 'NES' if 'NES' in gsea_df.columns else 'log2fc'
    
    # auto-set threshold based on score type
    if nes_thresh is None:
        nes_thresh = 1.5 if score_col == 'NES' else gsea_df[score_col].abs().quantile(0.75)
    
    # rest of function unchanged...
    df = gsea_df.copy()
    df['-log10_FDR'] = -np.log10(df['padj'].replace(0, np.nextafter(0, 1)))
    df['-log10_FDR'] = df['-log10_FDR'].clip(upper=ymax)  # cap floored values
    df['Direction'] = np.where(df['NES'] >= 0, 'MHC-II High', 'MHC-II Low')

    fig, ax = plt.subplots(figsize=figsize)
    sns.scatterplot(
        data=df, x='NES', y='-log10_FDR', hue='Direction',
        palette={'MHC-II High': cmap_high_low[0], 'MHC-II Low': cmap_high_low[1]},
        edgecolor='none', alpha=0.9, ax=ax,
    )
    ax.axvline( nes_thresh, color='black', lw=1, ls='--')
    ax.axvline(-nes_thresh, color='black', lw=1, ls='--')
    ax.axhline(-np.log10(sig_thresh), color='black', lw=1, ls='--')

    # note capped values
    ax.text(0.02, 0.98, f'Note: -log10(FDR) capped at {ymax} for display',
            transform=ax.transAxes, fontsize=8, va='top', color='gray')

    top_hits = df.sort_values('padj').head(top_n_labels)
    texts = []
    for _, row in top_hits.iterrows():
        texts.append(ax.text(
            row['NES'], row['-log10_FDR'],
            row.name.replace('_', ' ').title(),
            fontsize=8,
            ha='right' if row['NES'] < 0 else 'left',
        ))
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5), ax=ax)

    ax.set_xlabel('Normalized Enrichment Score (NES)')
    ax.set_ylabel(r'$-\log_{10}$(FDR)')
    ax.set_title(title or 'GSEA Volcano')
    ax.legend(title='Enriched In', bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig

In [ ]:
# Hallmark barplot
fig = plot_gsea_barplot(
    gsea_hallmark, top_n=50,
    title='Hallmark Pathways — MHC II pos vs neg (Salcher SCRNA)',
)
fig.savefig(supp_out / 'figS3_salcher_hallmark_gsea_barplot_ttest.pdf', bbox_inches='tight')
plt.show()

# Hallmark volcano
fig = plot_gsea_volcano(
    gsea_hallmark,
    title='Hallmark GSEA Volcano — MHC II pos vs neg (Salcher SCRNA)',
)
fig.savefig(supp_out / 'figS3_salcher_hallmark_gsea_volcano_ttest.pdf', bbox_inches='tight')
plt.show()

# CosMx pathways barplot
fig = plot_gsea_barplot(
    gsea_cosmx, top_n=50,
    title='CosMx Pathways — MHC II pos vs neg (Salcher SCRNA)',
)
fig.savefig(supp_out / 'figS3_salcher_cosmx_pathways_gsea_barplot_ttest.pdf', bbox_inches='tight')
plt.show()

# CosMx pathways volcano
fig = plot_gsea_volcano(
    gsea_cosmx,
    title='CosMx Pathways Volcano — MHC II pos vs neg (Salcher SCRNA)',
)
fig.savefig(supp_out / 'figS3_salcher_cosmx_pathways_gsea_volcano_ttest.pdf', bbox_inches='tight')
plt.show()